In [4]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install transformers scikit-learn pandas seqeval tqdm

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
import sys
print(sys.executable)

2.5.1+cu121
True
c:\Users\lusby\miniconda3\envs\torch-gpu\python.exe


In [1]:
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
  AutoTokenizer,
  AutoModelForSequenceClassification,
  get_linear_schedule_with_warmup
)
from sklearn.metrics import classification_report, f1_score
from collections import Counter, defaultdict
import random
import re
import os

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

c:\Users\lusby\miniconda3\envs\torch-gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Device: cuda
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design
Memory: 4.3 GB


In [2]:
RELATION_TYPES = [
  "no_relation",
  "causes",
  "increases",
  "decreases",
  "affects",
  "contributes_to",
  "occurs_in",
  "mitigates"
]

relation2id = {rel: i for i, rel in enumerate(RELATION_TYPES)}
id2relation = {i: rel for rel, i in relation2id.items()}

print(f"Total relation types: {len(RELATION_TYPES)}")
for rel, idx in relation2id.items():
  print(f" {idx}: {rel}")

Total relation types: 8
 0: no_relation
 1: causes
 2: increases
 3: decreases
 4: affects
 5: contributes_to
 6: occurs_in
 7: mitigates


In [3]:
def read_bio_file(filepath):
  sentences = []
  labels = []
  with open(filepath, "r", encoding="utf-8") as f:
    content = f.read()
  blocks = content.strip().split("\n\n")
  for block in blocks:
    block = block.strip()
    if not block:
      continue
    words = []
    tags = []
    for line in block.split("\n"):
      line = line.strip()
      if not line:
        continue
      parts = line.split("\t")
      if len(parts) == 2:
        words.append(parts[0])
        tags.append(parts[1])
    if words:
      sentences.append(words)
      labels.append(tags)
  return sentences, labels


def extract_entities(words, tags):
  entities = []
  current_entity = []
  current_label = None
  start_idx = 0
  for i, (word, tag) in enumerate(zip(words, tags)):
    if tag.startswith("B-"):
      if current_entity:
        entities.append({
          "text": " ".join(current_entity),
          "label": current_label,
          "start": start_idx,
          "end": i - 1
        })
      current_entity = [word]
      current_label = tag[2:]
      start_idx = i
    elif tag.startswith("I-") and current_entity:
      current_entity.append(word)
    else:
      if current_entity:
        entities.append({
          "text": " ".join(current_entity),
          "label": current_label,
          "start": start_idx,
          "end": i - 1
        })
      current_entity = []
      current_label = None
  if current_entity:
    entities.append({
      "text": " ".join(current_entity),
      "label": current_label,
      "start": start_idx,
      "end": len(words) - 1
    })
  return entities


def mark_entities(words, e1, e2):
  insertions = []
  insertions.append((e1["start"], "[E1]", "before"))
  insertions.append((e1["end"], "[/E1]", "after"))
  insertions.append((e2["start"], "[E2]", "before"))
  insertions.append((e2["end"], "[/E2]", "after"))
  result = []
  for i, word in enumerate(words):
    for pos, marker, timing in insertions:
      if pos == i and timing == "before":
        result.append(marker)
    result.append(word)
    for pos, marker, timing in insertions:
      if pos == i and timing == "after":
        result.append(marker)
  return " ".join(result)


def create_entity_pairs(sentences, labels):
  pairs = []
  for words, tags in zip(sentences, labels):
    entities = extract_entities(words, tags)
    if len(entities) < 2:
      continue
    for i in range(len(entities)):
      for j in range(len(entities)):
        if i == j:
          continue
        e1 = entities[i]
        e2 = entities[j]
        marked = mark_entities(words, e1, e2)
        pairs.append({
          "sentence": marked,
          "e1_text": e1["text"],
          "e1_label": e1["label"],
          "e2_text": e2["text"],
          "e2_label": e2["label"],
          "relation": "no_relation"
        })
  return pairs


# Update these paths to where your files are on your friend's laptop
BASE = r"C:\Climate_Intelligent_System\data\processed" # UPDATE THIS PATH

train_sentences, train_labels = read_bio_file(
  os.path.join(BASE, "climate_train.txt")
)

dev_sentences, dev_labels = read_bio_file(
  os.path.join(BASE, "climate_dev.txt")
)

test_sentences, test_labels = read_bio_file(
  os.path.join(BASE, "climate_test.txt")
)

print("Extracting entity pairs...")
train_pairs = create_entity_pairs(train_sentences, train_labels)
dev_pairs = create_entity_pairs(dev_sentences, dev_labels)
test_pairs = create_entity_pairs(test_sentences, test_labels)

print(f"Train pairs: {len(train_pairs)}")
print(f"Dev pairs: {len(dev_pairs)}")
print(f"Test pairs: {len(test_pairs)}")

Extracting entity pairs...
Train pairs: 115274
Dev pairs: 11456
Test pairs: 18490


In [4]:
def standardize_entity_label(label):
  mapping = {
    "CLIMATE_DRIVER": "Climate_Driver",
    "CLIMATE_VARIABLE": "Climate_Variable",
    "ENVIRONMENTAL_EVENT": "Env_Event",
    "ECOSYSTEM": "Ecosystem",
    "GEOGRAPHIC_LOCATION": "Geo_Location",
    "HUMAN_ACTIVITY": "Human_Activity",
    "POLICY": "Policy",
    "SPECIES": "Species",
  }
  return mapping.get(label, label)

RELATION_RULES = {
  ("Climate_Driver", "Climate_Variable"): "increases",
  ("Climate_Driver", "Env_Event"): "causes",
  ("Climate_Driver", "Ecosystem"): "affects",
  ("Climate_Driver", "Species"): "affects",
  ("Climate_Variable", "Env_Event"): "causes",
  ("Climate_Variable", "Ecosystem"): "affects",
  ("Climate_Variable", "Species"): "affects",
  ("Env_Event", "Ecosystem"): "affects",
  ("Env_Event", "Species"): "affects",
  ("Env_Event", "Geo_Location"): "occurs_in",
  ("Human_Activity", "Climate_Driver"): "contributes_to",
  ("Human_Activity", "Env_Event"): "causes",
  ("Human_Activity", "Ecosystem"): "affects",
  ("Policy", "Climate_Driver"): "mitigates",
  ("Policy", "Human_Activity"): "mitigates",
  ("Ecosystem", "Geo_Location"): "occurs_in",
  ("Species", "Ecosystem"): "occurs_in",
  ("Climate_Variable", "Climate_Variable"): "increases",
  ("Climate_Driver", "Climate_Driver"): "contributes_to",
}
KEYWORD_RULES = {
  "increases": ["increase", "increases", "increased", "rise",
  "rising", "higher", "amplif", "enhance"],
  "decreases": ["decrease", "decreases", "decreased", "decline",
  "declining", "reduce", "reduction", "lower"],
  "causes": ["cause", "causes", "caused", "lead to", "leads to",
  "result in", "results in", "trigger", "induce"],
  "affects": ["affect", "affects", "affected", "impact",
  "impacts", "influence", "threatens"],
  "contributes_to": ["contributes", "contribute", "contributes to",
  "drives", "driven by", "emit", "release"],
  "occurs_in": ["occurs in", "found in", "located in", "in the",
  "across", "throughout", "within"],
  "mitigates": ["mitigate", "mitigates", "reduce", "prevent",
  "offset", "sequester", "absorb"],
}


def assign_relation(pair):
  e1_label = standardize_entity_label(pair["e1_label"])
  e2_label = standardize_entity_label(pair["e2_label"])
  sentence = pair["sentence"].lower()
  for relation, keywords in KEYWORD_RULES.items():
    if any(kw in sentence for kw in keywords):
      type_pair = (e1_label, e2_label)
      if type_pair in RELATION_RULES:
        if RELATION_RULES[type_pair] == relation:
          return relation
  type_pair = (e1_label, e2_label)
  if type_pair in RELATION_RULES:
    return RELATION_RULES[type_pair]
  return "no_relation"


def apply_distant_supervision(pairs):
  for pair in pairs:
    pair["relation"] = assign_relation(pair)
  return pairs


def balance_dataset(pairs, max_per_class=2000):
  """
  Limit per class — 2000 is good for 4GB GPU
  """
  grouped = defaultdict(list)
  for pair in pairs:
    grouped[pair["relation"]].append(pair)
  
  balanced = []
  for rel, rel_pairs in grouped.items():
    random.shuffle(rel_pairs)
    balanced.extend(rel_pairs[:max_per_class])

  random.shuffle(balanced)
  
  print("Distribution:")
  counts = Counter(p["relation"] for p in balanced)
  for rel, count in sorted(
    counts.items(), key=lambda x: x[1], reverse=True
  ):
    print(f" {rel:20s}: {count}")
  print(f"Total: {len(balanced)}")
  return balanced

print("Applying distant supervision...")
train_pairs = apply_distant_supervision(train_pairs)
dev_pairs = apply_distant_supervision(dev_pairs)
test_pairs = apply_distant_supervision(test_pairs)

print("\nTrain:")
train_balanced = balance_dataset(train_pairs, max_per_class=2000)
print("\nDev:")
dev_balanced = balance_dataset(dev_pairs, max_per_class=500)
print("\nTest:")
test_balanced = balance_dataset(test_pairs, max_per_class=500)

Applying distant supervision...

Train:
Distribution:
 mitigates           : 2000
 increases           : 2000
 no_relation         : 2000
 occurs_in           : 2000
 contributes_to      : 2000
 affects             : 2000
 causes              : 2000
Total: 14000

Dev:
Distribution:
 no_relation         : 500
 occurs_in           : 500
 affects             : 500
 causes              : 500
 increases           : 500
 mitigates           : 309
 contributes_to      : 283
Total: 3092

Test:
Distribution:
 mitigates           : 500
 increases           : 500
 affects             : 500
 occurs_in           : 500
 no_relation         : 500
 causes              : 500
 contributes_to      : 483
Total: 3483


In [5]:
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

special_tokens = ["[E1]", "[/E1]", "[E2]", "[/E2]"]
tokenizer.add_special_tokens(
  {"additional_special_tokens": special_tokens}
)
print(f"Tokenizer loaded! Vocab size: {len(tokenizer)}")

class REDataset(Dataset):
  def __init__(self, pairs, tokenizer, max_length=128):
    self.pairs = pairs
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.pairs)

  def __getitem__(self, idx):
    pair = self.pairs[idx]
    encoding = self.tokenizer(
      pair["sentence"],
      max_length=self.max_length,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
    )
    return {
      "input_ids": encoding["input_ids"].squeeze(),
      "attention_mask": encoding["attention_mask"].squeeze(),
      "labels": torch.tensor(
        relation2id[pair["relation"]],
        dtype=torch.long
      )
    }

train_dataset = REDataset(train_balanced, tokenizer)
dev_dataset = REDataset(dev_balanced, tokenizer)
test_dataset = REDataset(test_balanced, tokenizer)

print(f"Train: {len(train_dataset)} samples")
print(f"Dev: {len(dev_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

c:\Users\lusby\miniconda3\envs\torch-gpu\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokenizer loaded! Vocab size: 31094
Train: 14000 samples
Dev: 3092 samples
Test: 3483 samples


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
  model_name,
  num_labels=len(RELATION_TYPES),
  ignore_mismatched_sizes=True
)
model.resize_token_embeddings(len(tokenizer))
model = model.to(device)

print(f"Model loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Check GPU memory
print(f"GPU memory allocated: "
  f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on cuda
Parameters: 109,927,688
GPU memory allocated: 0.44 GB


In [7]:
# Optimized for 4GB GPU
EPOCHS = 5
BATCH_SIZE = 16 # Safe for 4GB GPU
LEARNING_RATE = 2e-5

train_loader = DataLoader(
  train_dataset, batch_size=BATCH_SIZE,
  shuffle=True, num_workers=0
)
dev_loader = DataLoader(
  dev_dataset, batch_size=BATCH_SIZE,
  shuffle=False, num_workers=0
)
test_loader = DataLoader(
  test_dataset, batch_size=BATCH_SIZE,
  shuffle=False, num_workers=0
)

optimizer = AdamW(
  model.parameters(),
  lr=LEARNING_RATE,
  weight_decay=0.01
)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
  optimizer,
  num_warmup_steps=100,
  num_training_steps=total_steps
)

def evaluate_re(model, data_loader, device):
  model.eval()
  all_preds = []
  all_labels = []
  total_loss = 0
  with torch.no_grad():
    for batch in data_loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)
      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
      )
      total_loss += outputs.loss.item()
      predictions = torch.argmax(outputs.logits, dim=1)
      all_preds.extend(predictions.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())
  target_relations = [r for r in RELATION_TYPES
              if r != "no_relation"]
  f1 = f1_score(
    all_labels, all_preds,
    labels=[relation2id[r] for r in target_relations],
    average="macro",
    zero_division=0
  )
  return f1, total_loss/len(data_loader), all_preds, all_labels


print("Starting RE training...")
print(f"Train batches: {len(train_loader)}")
print("=" * 60)

best_dev_f1 = 0
best_epoch = 0

for epoch in range(EPOCHS):
  model.train()
  total_loss = 0
  
  for batch_idx, batch in enumerate(train_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()
    outputs = model(
      input_ids=input_ids,
      attention_mask=attention_mask,
      labels=labels
    )
    loss = outputs.loss
    total_loss += loss.item()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if (batch_idx + 1) % 100 == 0:
      print(f" Epoch {epoch+1} | "
      f"Batch {batch_idx+1}/{len(train_loader)} | "
      f"Loss: {total_loss/(batch_idx+1):.4f}")

avg_loss = total_loss / len(train_loader)
dev_f1, dev_loss, _, _ = evaluate_re(
  model, dev_loader, device
)

print(f"\nEpoch {epoch+1}/{EPOCHS}")
print(f" Train Loss: {avg_loss:.4f}")
print(f" Dev Loss: {dev_loss:.4f}")
print(f" Dev F1: {dev_f1:.4f}")

if dev_f1 > best_dev_f1:
  best_dev_f1 = dev_f1
  best_epoch = epoch + 1
  torch.save(
    model.state_dict(),
    "best_scibert_re.pt"
  )
  print(f" ✓ Best model saved!")
print("-" * 40)

print(f"\nTraining complete!")
print(f"Best Dev F1: {best_dev_f1:.4f} at epoch {best_epoch}")

Starting RE training...
Train batches: 875
 Epoch 1 | Batch 100/875 | Loss: 2.0067
 Epoch 1 | Batch 200/875 | Loss: 1.9137
 Epoch 1 | Batch 300/875 | Loss: 1.8730
 Epoch 1 | Batch 400/875 | Loss: 1.8194
 Epoch 1 | Batch 500/875 | Loss: 1.7520
 Epoch 1 | Batch 600/875 | Loss: 1.7019
 Epoch 1 | Batch 700/875 | Loss: 1.6472
 Epoch 1 | Batch 800/875 | Loss: 1.6050
 Epoch 2 | Batch 100/875 | Loss: 1.1451
 Epoch 2 | Batch 200/875 | Loss: 1.1398
 Epoch 2 | Batch 300/875 | Loss: 1.1282
 Epoch 2 | Batch 400/875 | Loss: 1.1291
 Epoch 2 | Batch 500/875 | Loss: 1.1141
 Epoch 2 | Batch 600/875 | Loss: 1.0994
 Epoch 2 | Batch 700/875 | Loss: 1.0858
 Epoch 2 | Batch 800/875 | Loss: 1.0795
 Epoch 3 | Batch 100/875 | Loss: 0.9285
 Epoch 3 | Batch 200/875 | Loss: 0.9383
 Epoch 3 | Batch 300/875 | Loss: 0.9315
 Epoch 3 | Batch 400/875 | Loss: 0.9331
 Epoch 3 | Batch 500/875 | Loss: 0.9229
 Epoch 3 | Batch 600/875 | Loss: 0.9149
 Epoch 3 | Batch 700/875 | Loss: 0.9178
 Epoch 3 | Batch 800/875 | Loss: 0.91

In [9]:
model.load_state_dict(torch.load("best_scibert_re.pt"))
test_f1, test_loss, all_preds, all_labels = evaluate_re(
  model, test_loader, device
)

print("=" * 60)
print("FINAL RE TEST RESULTS")
print("=" * 60)
print(f"Test F1: {test_f1:.4f}")
print(f"Test Loss: {test_loss:.4f}")
print("\nPer Relation Results:")
print(classification_report(
  all_labels, all_preds,
  labels=list(range(len(RELATION_TYPES))),
  target_names=RELATION_TYPES,
  zero_division=0
))

# Save model properly
model.save_pretrained("scibert_re_climate")
tokenizer.save_pretrained("scibert_re_climate")
print("\nModel saved to scibert_re_climate/")

C:\Users\lusby\AppData\Local\Temp\ipykernel_29572\409733196.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_scibert_re.pt"))


FINAL RE TEST RESULTS
Test F1: 0.4118
Test Loss: 1.8026

Per Relation Results:
                precision    recall  f1-score   support

   no_relation       0.28      0.29      0.29       500
        causes       0.61      0.45      0.52       500
     increases       0.38      0.37      0.38       500
     decreases       0.00      0.00      0.00         0
       affects       0.60      0.53      0.56       500
contributes_to       0.27      0.44      0.34       483
     occurs_in       0.78      0.47      0.59       500
     mitigates       0.46      0.54      0.50       500

      accuracy                           0.44      3483
     macro avg       0.42      0.39      0.40      3483
  weighted avg       0.49      0.44      0.45      3483


Model saved to scibert_re_climate/
